In [1]:
import os
import re
import time
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

In [2]:
MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 256
NORMALIZE_EMBEDDINGS = True

INPUT_PATH = Path(r"D:\nih_data\processed\corpus\corpus.parquet") 

OUTPUT_DIR = Path(r"D:\nih_data\processed\embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_FILE = OUTPUT_DIR / "embeddings.npy"
INDEX_FILE = OUTPUT_DIR / "embedding_index.parquet"

In [3]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)
df.head()

(403988, 10)


,APPLICATION_ID,FY,IC_NAME,ACTIVITY,PROJECT_TITLE,ABSTRACT_TEXT,PROJECT_TERMS,NIH_SPENDING_CATS,TOTAL_COST,text
0,8676197,2020,Veterans Affairs,IK2,Improving Care for PTSD,DESCRIPTION (provided by applicant): BACK...,Affect;Area;Award;career;career development;Ca...,None,NaN,Improving Care for PTSD\n\nDESCRIPTION (provid...
1,8785548,2020,Veterans Affairs,I01,Design and Evaluation of User Centered Electro...,DESCRIPTION (provided by applicant): To d...,acronyms;Address;Affect;Archives;Area;Automati...,None,NaN,Design and Evaluation of User Centered Electro...
2,8867710,2020,Veterans Affairs,I01,Comparative Effectiveness of Delivery Methods ...,? DESCRIPTION (provided by applicant): ...,18 year old;Address;Adult;Adult Children;Age;A...,None,NaN,Comparative Effectiveness of Delivery Methods ...
3,8900803,2020,Veterans Affairs,I01,Relational Agent to Improve Alcohol Screening ...,Background: VA has demonstrated national leade...,Acute;Address;Age;Alcohol abuse;alcohol abuse ...,None,NaN,Relational Agent to Improve Alcohol Screening ...
4,8978569,2020,Veterans Affairs,I01,Assessing Hypertension Care for Aged Veterans:...,? DESCRIPTION (provided by applicant): ...,acute stroke;Adherence;Affect;Age;age group;ag...,None,NaN,Assessing Hypertension Care for Aged Veterans:...


In [4]:
required_cols = ["APPLICATION_ID", "FY", "text"]
assert all(col in df.columns for col in required_cols)

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return None
    text = re.sub(r"\s+", " ", text) 
    return text.strip()

In [6]:
df["text_norm"] = df["text"].apply(normalize_text)
df = df[df["text_norm"].notnull()]
df = df[df["text_norm"].str.len() > 0]

In [7]:
def hash_text(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

df["text_hash"] = df["text_norm"].apply(hash_text)

In [8]:
model = SentenceTransformer(MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Embedding dimension: 384


In [9]:
def embed_texts(texts, batch_size=256):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_embeddings = model.encode(
            batch,
            show_progress_bar=False,
            convert_to_numpy=True
        )
        embeddings.append(batch_embeddings)
    
    return np.vstack(embeddings)

In [11]:
start = time.time()

embeddings = embed_texts(
    df["text_norm"].tolist(),
    batch_size=BATCH_SIZE
)

if NORMALIZE_EMBEDDINGS:
    embeddings = normalize(embeddings)

print("Time:", round(time.time() - start, 2), "seconds")
print("Shape:", embeddings.shape)

  1%|██▍                                                                                                                                                                              | 22/1579 [04:13<4:58:33, 11.51s/it]


KeyboardInterrupt: 

In [ ]:
assert embeddings.shape[0] == len(df)

In [ ]:
np.save(EMBEDDING_FILE, embeddings)
index_df = df[[
    "APPLICATION_ID",
    "FY",
    "IC_NAME",
    "ACTIVITY",
    "TOTAL_COST",
    "text_hash"
]].reset_index(drop=True)

index_df.to_parquet(INDEX_FILE, index=False)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sample_idx = 10
sims = cosine_similarity(
    embeddings[sample_idx].reshape(1, -1),
    embeddings
)[0]

top_idx = np.argsort(sims)[-6:][::-1]
df.iloc[top_idx][["PROJECT_TITLE", "FY"]]

In [ ]:
metadata = {
    "model": MODEL_NAME,
    "embedding_dim": embeddings.shape[1],
    "n_documents": len(df),
    "batch_size": BATCH_SIZE,
    "normalized": NORMALIZE_EMBEDDINGS,
    "timestamp": pd.Timestamp.now()
}

pd.Series(metadata).to_json(OUTPUT_DIR / "embedding_metadata.json")